# 00 — Common code
Diğer tüm notebook'lar bunu `%run 00_common.ipynb` ile çalıştırır. Burada sadece **tanımlar** var (config, dataset, model, eğitim/değerlendirme fonksiyonları); hiçbir şey eğitilmez.

In [ ]:
import json, os, random, shutil, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

## Config

In [ ]:
IN_COLAB = "google.colab" in sys.modules
ROOT_DIR = Path.cwd().parent  # notebooks/ is the working directory

# Raw images: on Colab keep them on the local disk (fast), not on Drive
RAW_DATA_DIR = Path("/content/data") if IN_COLAB else ROOT_DIR / "data"
TRAIN_CSV = RAW_DATA_DIR / "boneage-training-dataset.csv"
TRAIN_IMG_DIR = RAW_DATA_DIR / "boneage-training-dataset" / "boneage-training-dataset"
CACHE_SIZE = 640  # raw images (~2000px) are resized once to speed up loading
CACHE_DIR = RAW_DATA_DIR / f"resized_{CACHE_SIZE}"

# Small files live inside the project folder (on Drive in Colab) so they survive disconnects
SPLITS_CSV = ROOT_DIR / "data" / "splits.csv"
OUTPUT_DIR = ROOT_DIR / "outputs"
for d in (RAW_DATA_DIR, SPLITS_CSV.parent, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_FRAC, TEST_FRAC = 0.10, 0.10
N_AGE_BINS = 10

IMG_SIZE = 384
BATCH_SIZE = 16
NUM_WORKERS = 0 if os.name == "nt" else 2  # Windows can't send notebook-defined classes to worker processes

BACKBONE = "efficientnet_b0"
SEX_DIM, HIDDEN_DIM, DROPOUT = 16, 256, 0.2

EPOCHS = 30
LR_HEAD, LR_BACKBONE = 1e-3, 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Image cache

In [ ]:
def _resize_one(img_id):
    out_path = CACHE_DIR / f"{img_id}.png"
    if out_path.exists():
        return
    img = cv2.imread(str(TRAIN_IMG_DIR / f"{img_id}.png"), cv2.IMREAD_GRAYSCALE)
    scale = CACHE_SIZE / max(img.shape)
    if scale < 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(out_path), img)


def cache_images(ids):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(tqdm(pool.map(_resize_one, ids), total=len(ids), desc="Caching images"))

## Split + baselines

In [ ]:
def make_splits(df):
    """Stratify by (boneage bin, sex) so every split has a similar distribution."""
    age_bin = pd.qcut(df["boneage"], q=N_AGE_BINS, labels=False, duplicates="drop")
    strat = age_bin.astype(str) + "_" + df["male"].astype(str)
    train_df, rest_df = train_test_split(df, test_size=VAL_FRAC + TEST_FRAC, stratify=strat, random_state=SEED)
    val_df, test_df = train_test_split(rest_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                                       stratify=strat.loc[rest_df.index], random_state=SEED)
    df = df.copy()
    df["split"] = "train"
    df.loc[val_df.index, "split"] = "val"
    df.loc[test_df.index, "split"] = "test"
    return df


def baselines(df):
    """Mean and per-sex mean predictions (fit on train)."""
    train = df[df["split"] == "train"]
    global_mean = train["boneage"].mean()
    sex_means = train.groupby("male")["boneage"].mean()
    rows = []
    for split in ["val", "test"]:
        part = df[df["split"] == split]
        rows.append({"split": split,
                     "mean_baseline_mae": mean_absolute_error(part["boneage"], np.full(len(part), global_mean)),
                     "sex_mean_baseline_mae": mean_absolute_error(part["boneage"], part["male"].map(sex_means))})
    return pd.DataFrame(rows)

## Dataset + augmentation
**Data augmentation (veri çoğaltma):** Eğitim sırasında görüntüye rastgele küçük değişiklikler (döndürme, kaydırma, parlaklık) uygulanır. Model her epoch'ta biraz farklı görüntü görür → ezberlemek (overfitting) zorlaşır. Sadece train setine uygulanır.

In [ ]:
def get_transforms(train, img_size=IMG_SIZE, extra=()):
    resize = [
        A.LongestMaxSize(max_size=img_size),  # keep aspect ratio...
        A.PadIfNeeded(img_size, img_size, border_mode=cv2.BORDER_CONSTANT, fill=0),  # ...and pad to a square
    ]
    augment = [
        A.Affine(rotate=(-15, 15), translate_percent=(-0.05, 0.05), scale=(0.9, 1.1),
                 border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.8),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.CLAHE(clip_limit=2.0, p=0.3),
    ] if train else []
    normalize = [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return A.Compose(resize + augment + list(extra) + normalize)


class BoneAgeDataset(Dataset):
    """Returns (image [3,H,W], sex [1], normalized target [1])."""

    def __init__(self, df, transform, target_mean, target_std):
        self.ids = df["id"].values
        self.sex = df["male"].astype(np.float32).values
        self.target_mean, self.target_std = target_mean, target_std
        self.target = ((df["boneage"].values - target_mean) / target_std).astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img = cv2.imread(str(CACHE_DIR / f"{self.ids[idx]}.png"), cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  # 3 channels for the ImageNet-pretrained backbone
        img = self.transform(image=img)["image"]
        return img, torch.tensor([self.sex[idx]]), torch.tensor([self.target[idx]])


def load_splits():
    """Splits + target mean/std computed on train only (no leakage)."""
    df = pd.read_csv(SPLITS_CSV)
    train = df[df["split"] == "train"]
    return df, (train["boneage"].mean(), train["boneage"].std())


def make_loader(df, split, stats, img_size=IMG_SIZE, batch_size=BATCH_SIZE, transform=None):
    train = split == "train"
    part = df[df["split"] == split].reset_index(drop=True)
    ds = BoneAgeDataset(part, transform or get_transforms(train, img_size), *stats)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
                      drop_last=train, persistent_workers=train and NUM_WORKERS > 0)


def show_batch(loader, n=8):
    imgs, sexes, targets = next(iter(loader))
    ds = loader.dataset
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
    for i, ax in enumerate(axes):
        img = imgs[i].permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(f"{targets[i].item() * ds.target_std + ds.target_mean:.0f} mo, {'M' if sexes[i].item() else 'F'}")
        ax.axis("off")
    plt.tight_layout()
    return fig

## Model
**Transfer learning (aktarım öğrenimi):** ImageNet'te milyonlarca görüntüyle önceden eğitilmiş bir ağ (EfficientNet) kenar, doku, şekil gibi genel görsel özellikleri zaten öğrenmiştir. Biz bu ağı alıp sınıflandırma katmanını çıkarıyor, yerine kendi regresyon "head"imizi takıp kemik yaşına göre ince ayar (fine-tuning) yapıyoruz. Sıfırdan eğitmeye göre çok daha az veri ve zamanla çok daha iyi sonuç verir.

Cinsiyet küçük bir `Linear(1, 16)` katmanından geçip görüntü özellikleriyle birleştirilir (concat).

In [ ]:
class BoneAgeModel(nn.Module):
    def __init__(self, backbone=BACKBONE, pretrained=True):
        super().__init__()
        # num_classes=0 removes the ImageNet classifier; we keep the pooled features
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        self.sex_fc = nn.Sequential(nn.Linear(1, SEX_DIM), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features + SEX_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, 1),
        )

    def forward_from_features(self, feature_map, sex):
        """Pool the backbone feature map and predict (split out so Grad-CAM can reuse it)."""
        img_feat = self.backbone.forward_head(feature_map, pre_logits=True)
        return self.head(torch.cat([img_feat, self.sex_fc(sex)], dim=1))

    def forward(self, img, sex):
        return self.forward_from_features(self.backbone.forward_features(img), sex)

## Training
- **L1Loss:** Ortalama mutlak hata; doğrudan hedef metriğimiz MAE'yi optimize eder.
- **AdamW + iki farklı learning rate:** Yeni eklenen head rastgele başladığı için büyük lr (1e-3), önceden eğitilmiş backbone'u bozmamak için küçük lr (1e-4).
- **CosineAnnealingLR (öğrenme oranı zamanlayıcı):** lr'yi kosinüs eğrisiyle yavaşça sıfıra indirir; sonlara doğru küçük adımlarla ince ayar yapılır.
- **Early stopping (erken durdurma):** Val MAE 5 epoch üst üste iyileşmezse durur → overfitting'i ve boşa harcanan zamanı önler.
- **Mixed precision (`torch.amp`):** Hesapların çoğu float16 ile yapılır → GPU'da ~2x hız, daha az bellek.

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, loss_fn):
    model.train()
    total = 0.0
    for img, sex, target in tqdm(loader, desc="train", leave=False):
        img, sex, target = img.to(DEVICE), sex.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = loss_fn(model(img, sex), target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(img)
    return total / len(loader.dataset)


@torch.no_grad()
def predict(model, loader, stats):
    """Predictions in months (de-normalized)."""
    model.eval()
    preds = []
    for img, sex, _ in tqdm(loader, desc="predict", leave=False):
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            preds.append(model(img.to(DEVICE), sex.to(DEVICE)).float().cpu())
    return torch.cat(preds).squeeze(1).numpy() * stats[1] + stats[0]


def train_model(run_name, backbone=BACKBONE, img_size=IMG_SIZE, batch_size=BATCH_SIZE, epochs=EPOCHS):
    """Train one model; saves best checkpoint + per-epoch log to OUTPUT_DIR/run_name."""
    set_seed()
    df, stats = load_splits()
    train_loader = make_loader(df, "train", stats, img_size, batch_size)
    val_loader = make_loader(df, "val", stats, img_size, batch_size)
    val_true = df[df["split"] == "val"]["boneage"].values

    model = BoneAgeModel(backbone).to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": list(model.sex_fc.parameters()) + list(model.head.parameters()), "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler()
    loss_fn = nn.L1Loss()

    run_dir = OUTPUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_mae, bad_epochs, log = float("inf"), 0, []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, loss_fn)
        val_mae = mean_absolute_error(val_true, predict(model, val_loader, stats))
        scheduler.step()

        log.append({"epoch": epoch, "train_mae_months": train_loss * stats[1], "val_mae_months": val_mae,
                    "seconds": time.time() - t0})
        pd.DataFrame(log).to_csv(run_dir / "log.csv", index=False)
        improved = val_mae < best_mae
        if improved:
            best_mae, bad_epochs = val_mae, 0
            torch.save({"model": model.state_dict(), "stats": stats, "backbone": backbone,
                        "img_size": img_size, "epoch": epoch, "val_mae": val_mae}, run_dir / "best_model.pth")
        else:
            bad_epochs += 1
        print(f"epoch {epoch:02d} | train MAE {train_loss * stats[1]:.2f} | val MAE {val_mae:.2f} mo"
              f" | {time.time() - t0:.0f}s {'*' if improved else ''}")
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
    print(f"Best val MAE: {best_mae:.2f} months")
    return best_mae


def plot_curves(run_name):
    log = pd.read_csv(OUTPUT_DIR / run_name / "log.csv")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(log["epoch"], log["train_mae_months"], "o-", label="train MAE (with augmentation)")
    ax.plot(log["epoch"], log["val_mae_months"], "o-", label="val MAE")
    ax.set(xlabel="epoch", ylabel="MAE (months)", title=f"Learning curves — {run_name}")
    ax.legend()
    fig.savefig(OUTPUT_DIR / run_name / "learning_curves.png", dpi=120, bbox_inches="tight")
    return fig

## Evaluation
**TTA (Test Time Augmentation):** Tahmin sırasında aynı görüntünün birkaç hafif değiştirilmiş halini (burada ±7° döndürme) modele verip tahminlerin ortalamasını alırız → daha kararlı tahmin.
**Ensemble:** Farklı modellerin tahminlerinin ortalaması; modellerin hataları kısmen birbirini götürür.

In [ ]:
AGE_BINS = [0, 60, 120, 180, 1000]
AGE_LABELS = ["0-5y", "5-10y", "10-15y", "15+y"]


def tta_transforms(img_size):
    """Original + small deterministic rotations."""
    return [get_transforms(False, img_size)] + [
        get_transforms(False, img_size, extra=[
            A.Affine(rotate=(angle, angle), border_mode=cv2.BORDER_CONSTANT, fill=0, p=1.0)])
        for angle in (-7, 7)
    ]


def load_model(run_name):
    ckpt = torch.load(OUTPUT_DIR / run_name / "best_model.pth", map_location=DEVICE, weights_only=False)
    model = BoneAgeModel(ckpt["backbone"], pretrained=False).to(DEVICE).eval()
    model.load_state_dict(ckpt["model"])
    return model, ckpt


def predict_run(run_name, split, tta=False):
    model, ckpt = load_model(run_name)
    df, _ = load_splits()
    transforms = tta_transforms(ckpt["img_size"]) if tta else [get_transforms(False, ckpt["img_size"])]
    preds = [predict(model, make_loader(df, split, ckpt["stats"], batch_size=32, transform=t), ckpt["stats"])
             for t in transforms]
    return np.mean(preds, axis=0)


def evaluate(runs, split="val", tta=False):
    """Mean prediction of the given runs (1 run = single model, more = ensemble)."""
    df, _ = load_splits()
    res = df[df["split"] == split].reset_index(drop=True)
    res["pred"] = np.mean([predict_run(r, split, tta) for r in runs], axis=0)
    res["error"] = res["pred"] - res["boneage"]
    res["abs_error"] = res["error"].abs()
    res["age_group"] = pd.cut(res["boneage"], AGE_BINS, labels=AGE_LABELS, right=False)
    metrics = {"mae": res["abs_error"].mean(),
               "rmse": float(np.sqrt(mean_squared_error(res["boneage"], res["pred"]))),
               "r2": r2_score(res["boneage"], res["pred"])}
    return res, metrics


def plot_results(res, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(res["boneage"], res["pred"], s=8, alpha=0.5)
    lim = [0, max(res["boneage"].max(), res["pred"].max()) + 10]
    ax.plot(lim, lim, "r--", label="y = x")
    ax.set(xlim=lim, ylim=lim, xlabel="True bone age (months)", ylabel="Predicted (months)",
           title=f"Predicted vs true (MAE {res['abs_error'].mean():.2f} mo)")
    ax.legend()
    fig.savefig(out_dir / "scatter.png", dpi=120, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(res["error"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(0, color="r", linestyle="--")
    ax.set(xlabel="Error = predicted - true (months)", ylabel="Count", title="Error distribution")
    fig.savefig(out_dir / "error_hist.png", dpi=120, bbox_inches="tight")
    plt.show()


def show_worst(res, out_dir, n=10):
    worst = res.nlargest(n, "abs_error")
    fig, axes = plt.subplots(2, n // 2, figsize=(16, 8))
    for ax, (_, row) in zip(axes.ravel(), worst.iterrows()):
        ax.imshow(cv2.imread(str(CACHE_DIR / f"{row['id']}.png"), cv2.IMREAD_GRAYSCALE), cmap="gray")
        ax.set_title(f"id {row['id']} {'M' if row['male'] else 'F'}\ntrue {row['boneage']:.0f} / pred {row['pred']:.0f}")
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(out_dir / "worst10.png", dpi=100, bbox_inches="tight")
    plt.show()
    return worst

## Grad-CAM
Son konvolüsyon katmanının özellik haritalarını, tahmine göre gradyanlarının ortalamasıyla ağırlıklandırıp topluyoruz → modelin tahmin yaparken görüntünün hangi bölgelerine "baktığını" gösteren bir ısı haritası.

In [ ]:
def grad_cam(model, img, sex):
    """Weight each feature channel by its mean gradient, sum, ReLU -> heatmap in [0, 1]."""
    feature_map = model.backbone.forward_features(img)
    feature_map.retain_grad()
    pred = model.forward_from_features(feature_map, sex)
    model.zero_grad()
    pred.sum().backward()
    weights = feature_map.grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * feature_map).sum(dim=1)).squeeze(0).detach().cpu().numpy()
    cam = cv2.resize(cam, (img.shape[3], img.shape[2]))
    return cam / (cam.max() + 1e-8), pred.item()